# 3 — Path clusters

Loads the tables built by **notebook 1** and looks at the **path clustering** — how each trial's
trajectory is grouped into *Direct* vs *Corner-dwelling* (with an occasional *Exploratory* middle).
**It reads no logs and no video.**

The grouping itself is already done upstream (`task_*/cluster_paths.py`, run as a library inside
notebook 1): each trial is reduced to four PATH-GEOMETRY features — **efficiency, speed variability,
time in corner, trial length** — standardised, and KMeans-split; the highest-efficiency group is named
*Direct*, the lowest *Corner-dwelling*. Every trial already carries `cluster` / `cluster_name` and the
2-D PCA view (`cluster_pca1` / `cluster_pca2`). This notebook only **draws** that.

**Descriptive only** — it shows the grouping and the trajectories behind it, and draws no verdict. The
behavioural *consequences* of the split (whisking, joystick, heading) belong to the per-trial report,
not here.

Order: 1 load · 2 pick a task/animal · 3 the clusters (counts + defining features) · 4 the 2-D view ·
5 example trajectories · 6 composition by outcome / animal / world.

## Load

In [ ]:
MAIN_DIR = '/mnt/server/data'
PIPELINE_DIR = None
# =============================================================================
import sys, importlib
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

MAIN_DIR = Path(MAIN_DIR).expanduser()
cands = ([Path(PIPELINE_DIR)] if PIPELINE_DIR else []) + [
    Path.cwd().parent, Path.cwd(), Path.cwd().parent / 'session_pipeline']
PIPE = next((c.resolve() for c in cands if (c / 'common' / 'session_index.py').exists()), None)
sys.path.insert(0, str(PIPE / 'common')); sys.path.insert(0, str(PIPE / 'performance'))

import build_log_df as bl, plot_clusters as pc
bl = importlib.reload(bl); pc = importlib.reload(pc)

_LOCAL = Path('~/repo/session_pipeline_output').expanduser()
OUT = _LOCAL if (_LOCAL / 'df_trials.pkl').exists() else (MAIN_DIR / 'df_log')
print(f'loading dataset from {OUT}')
df_sessions = bl.load(OUT / 'df_sessions.pkl')
df_trials   = bl.load(OUT / 'df_trials.pkl')
print(f'\n{len(df_trials)} trials, {df_sessions.mouse.nunique()} animal(s)')
if 'cluster_name' in df_trials.columns:
    print('cluster_name values:', df_trials.cluster_name.value_counts(dropna=False).to_dict())
else:
    print('!! no cluster_name column -- rerun notebook 1 (the clustering step did not write it)')

## 1 — Pick a task / animal ← YOU SET THIS

Clusters are **per task** (a world fixes the geometry), so choose one task. `ANIMAL = None` pools every
animal of that task for the counts/features; the composition panel (step 5) can then split them back
apart by animal.

In [ ]:
TASK   = 'banish_multiplier'    # or 'timeout_multiplier'
ANIMAL = None                   # None = every animal of that task, or e.g. 'JPAS_0168'
# =============================================================================
T = df_trials[df_trials.task == TASK] if 'task' in df_trials.columns else df_trials
if ANIMAL: T = T[T.mouse == ANIMAL]
C = pc.clustered(T)             # real path clusters only (drops escape/degenerate/unclustered)
print(f'{len(T)} trials in the selection, {len(C)} carry a path cluster')
if len(C):
    print(C.cluster_name.value_counts().to_dict())
    print('  animals:', sorted(C.mouse.unique()) if 'mouse' in C.columns else 'n/a')
else:
    print('  nothing to plot -- this task may have no clustered trials in the dataset')

## 2 — The clusters: how many, and what defines them

One bar per cluster (trial count), then the four **path-geometry features** that produced the split.
*Direct* should read high efficiency + low corner-time + short; *Corner-dwelling* the opposite.

In [ ]:
fig = pc.cluster_overview(C, title=f'{TASK} - path clusters')
plt.show()

## 3 — The clusters in 2-D

Left: the stored **PCA** view of the 4-feature space (what KMeans actually split). Right: the raw
**efficiency vs time-in-corner** plane, so the grouping can be read in real units.

In [ ]:
fig = pc.cluster_scatter(C, title=f'{TASK} - cluster space')
plt.show()

## 4 — Example trajectories per cluster

A handful of real paths from each cluster — the point of the whole split. Each is colour-coded from
trial start (dark) to the collection (bright); black dot = start, star = collection.

In [ ]:
fig = pc.example_paths(C, n=5, title=f'{TASK} - example paths')
plt.show()

## 5 — Composition

What share of each group's trials is Direct vs Corner-dwelling — by **outcome** (do banishments come
from corner-dwelling paths?), by **animal**, and by **world**. Change `by=` to any column.

In [ ]:
for by in ['outcome', 'mouse', 'world']:
    if by in C.columns and C[by].nunique() > 1:
        fig = pc.cluster_composition(C, by=by, title=f'{TASK} - cluster share by {by}')
        plt.show()